# 18.1M Hindi–Sanskrit decoder

This size is **18,143,616** parameters. The command flag is `--model high`. The blueprint is `configs/model_high.yaml`. Context is 512 tokens, about 200 words, so a passage has to fit in that window.

The same blueprint was trained three times: high-only text, high plus medium text, and a short essay fine-tune on top of the second run. The sibling notebooks are `model_10m.ipynb` and `model_50m.ipynb`.


## Architecture

Same block as the 10.9M model: grouped-query attention, SwiGLU, RMSNorm, rotary base 100,000, tied embeddings.

| Piece | Setting |
| --- | --- |
| Vocabulary | 16,000 |
| Width | 384 |
| Layers | 8 |
| Query heads | 6 |
| Key-value heads | 2 |
| Head dimension | 64 |
| SwiGLU intermediate | 960 |
| Context | 512 |
| Dropout | 0.05 |

| Weights | Count |
| --- | --- |
| Tied token embedding, 16,000 × 384 | 6,144,000 |
| One block: attention 393,216 + SwiGLU 1,105,920 + two RMSNorm scales 768 | 1,499,904 |
| Eight blocks | 11,999,232 |
| Final RMSNorm | 384 |
| **Total** | **18,143,616** |

Width equals 6 × 64. Output projections start at `0.02 / sqrt(2 * 8)`.


In [1]:
from pathlib import Path
import sys
import yaml

HERE = Path.cwd()
ROOT = HERE if (HERE / "configs" / "model_10m.yaml").exists() else HERE / "experiments" / "language" / "010-hindi-sanskrit-slm"
sys.path.insert(0, str(ROOT))
print("experiment", ROOT)

CONFIG_NAME = "configs/model_high.yaml"
EXPECTED = 18_143_616

from src.model import build_model, count_parameters

config = yaml.safe_load((ROOT / CONFIG_NAME).read_text())
counted = count_parameters(config)
model = build_model(config)
live = model.num_parameters()
print("config", CONFIG_NAME)
print("formula", f"{counted:,}")
print("built", f"{live:,}")
print("expected", f"{EXPECTED:,}")
print("tied", model.lm_head.weight.data_ptr() == model.wte.weight.data_ptr())
print(
    "layers", len(model.blocks),
    "width", config["n_embd"],
    "query heads", config["n_head"],
    "kv heads", config["n_kv_head"],
    "head dim", config["head_dim"],
    "context", config["block_size"],
    "vocab", config["vocab_size"],
)
assert live == EXPECTED == counted


experiment /Users/akashchauhan/Development/ai-models-learning/experiments/language/010-hindi-sanskrit-slm
config configs/model_high.yaml
formula 18,143,616
built 18,143,616
expected 18,143,616
tied True
layers 8 width 384 query heads 6 kv heads 2 head dim 64 context 512 vocab 16000


## Data

Wikipedia stayed out. IN22, FLORES, and the official Samasāmayik test stayed out of training and were used only as benchmarks.

| Run | Training text | Tokenizer |
| --- | --- | --- |
| High only | Samasāmayik, both directions. Built by `scripts/build_high_only.py` into `data/clean-high/` | `tokenizers/sp_high.model`, 16,000 pieces |
| High and medium | Samasāmayik plus the English-pivoted BPCC pairs. Built by `scripts/build_high_medium.py` into `data/clean-high-medium/` | `tokenizers/sp_hm.model`, a new 16,000-piece model, because the alphabet grew |
| Essay | MIDAS public-domain Hindi stories, Devanagari Mahābhārata and Rāmāyaṇa cut into passages that fit in 512 tokens, plus a Samasāmayik replay. Built by `scripts/build_essay.py` | the high-and-medium tokenizer |

Those JSONL files and the `.model` tokenizers are local. They are not in git.


## What was trained

Each of the first two runs used 4 epochs, batch 64, the default pretrain peak 6e-4 and fine-tune peak 2e-4.

| Run | Pretrain tokens | Pretrain loss | Fine-tune tokens | Fine-tune loss |
| --- | --- | --- | --- | --- |
| High only | 12,427,520 | 5.56 | 11,959,587 | 3.14 |
| High and medium | 29,924,160 | 5.184 | 28,849,820 | 3.040 |
| Essay, started from the high-and-medium fine-tune | — | — | 2,981,180 | 4.70 |

Greedy chrF on held-out benchmarks:

| Benchmark | Copy | High only | High and medium |
| --- | --- | --- | --- |
| Samasāmayik official test | 32.30 | **39.07** | 24.95 |
| IN22 | 21.53 | 13.41 | 21.55 |
| FLORES | 21.70 | 15.35 | 21.95 |

High-only is the stronger in-domain translator. Adding the pivoted pairs lowered that score and moved the out-of-domain sets to about the copy baseline.

The essay fine-tune used peak learning rate 5e-5. Short translations such as “पानी ठंडा है” still become “जलं शीतलम् अस्ति”. A prompted essay still stops after a short incoherent stretch. Checkpoints: `runs/high-medium/sft/from-latest` and `runs/essay/sft/latest`. Cloud copies: `gs://niramana-ai-model-weights-dev/hindi-sanskrit-high/` and `gs://niramana-ai-model-weights-dev/hindi-sanskrit-high-medium/`.


In [1]:
CHECKPOINTS = [
    "runs/high-medium/sft/from-latest",
    "runs/essay/sft/latest",
]

from src.sample import continue_prompt, load_checkpoint

PROMPTS = [
    ("translate_hi_sa", "पानी ठंडा है।"),
    ("translate_hi_sa", "मैं किताब पढ़ता हूँ।"),
    ("translate_sa_hi", "जलं शीतलम् अस्ति।"),
]

def show(directory, task, prompt, max_new_tokens=32):
    path = ROOT / directory
    if not (path / "checkpoint.pt").exists():
        print(f"missing {directory}")
        return None
    text, checkpoint = continue_prompt(
        path, task, prompt, max_new_tokens=max_new_tokens, temperature=0.0,
        model=show.model, tokenizer=show.tokenizer, device=show.device,
    )
    body = text.split("<tgt>", 1)[-1].strip() if "<tgt>" in text else text.strip()
    print(f"{task}")
    print(f"  prompt: {prompt}")
    print(f"  output: {body}")
    return checkpoint

for directory in CHECKPOINTS:
    path = ROOT / directory
    print("\n#", directory)
    if not (path / "checkpoint.pt").exists():
        print("checkpoint is not on this machine")
        continue
    show.model, show.tokenizer, checkpoint, show.device = load_checkpoint(path)
    print("stage", checkpoint["stage"], "tokens", f"{checkpoint['tokens_seen']:,}")
    for task, prompt in PROMPTS:
        show(directory, task, prompt)



# runs/high-medium/sft/from-latest
stage sft tokens 28,849,820
translate_hi_sa
  prompt: पानी ठंडा है।
  output: जलं शीतलं भवति।
translate_hi_sa
  prompt: मैं किताब पढ़ता हूँ।
  output: अहं पुस्तकं पठामि।
translate_sa_hi
  prompt: जलं शीतलम् अस्ति।
  output: जल ठंडा है।

# runs/essay/sft/latest
stage sft tokens 2,981,180
translate_hi_sa
  prompt: पानी ठंडा है।
  output: जलं शीतलम् अस्ति ।
translate_hi_sa
  prompt: मैं किताब पढ़ता हूँ।
  output: अहं पुस्तकं पठामि।
translate_sa_hi
  prompt: जलं शीतलम् अस्ति।
  output: जल ठंडी है।


## Commands

```bash
uv run python -m src.smoke --model high
uv run python scripts/build_high_only.py
uv run python -m src.train --stage pretrain --model high --data data/clean-high/pretrain.jsonl --tokenizer tokenizers/sp_high.model --runs-dir runs/high --batch-size 64
uv run python -m src.train --stage sft --model high --init runs/high/pretrain/latest --data data/clean-high/sft.jsonl --tokenizer tokenizers/sp_high.model --runs-dir runs/high --batch-size 64
uv run python -m src.sample --checkpoint runs/high-medium/sft/from-latest --task translate_hi_sa --prompt "पानी ठंडा है।"
```
